# 01. EDA 결과를 MLflow run으로 남기기

책 4.3절. 데이터를 훑어본 결과(분포 플롯)를 노트북 안에만 두지 않고 MLflow run의 아티팩트로 남깁니다.
원본 데이터셋도 SeaweedFS에 올리고 `log_input`으로 run에 연결해, 이후 실험이 어느 데이터에서 출발했는지 추적할 수 있게 합니다.

In [ ]:
import os
import mlflow
import pandas as pd
from plotnine import aes, coord_flip, geom_bar, ggplot, labs, theme_bw
from income import CATEGORICAL, load_income_data, upload_df

# 접속 정보는 파드 환경 변수에 있다. 노트북 코드에는 주소와 계정을 적지 않는다.
print("MLFLOW_TRACKING_URI =", os.environ["MLFLOW_TRACKING_URI"])
print("AWS_ENDPOINT_URL    =", os.environ["AWS_ENDPOINT_URL"])

In [ ]:
df = load_income_data()
print(df.shape)
df.head()

In [ ]:
# 범주형 컬럼마다 값 분포와 Target 비율을 본다.
df.describe()

In [ ]:
os.makedirs("eda_plots", exist_ok=True)
for col in CATEGORICAL:
    plot = (ggplot(df) + geom_bar(aes(x=col, fill="Target"), position="fill")
            + theme_bw() + labs(title=f"{col} ~ Target") + coord_flip())
    plot.save(f"eda_plots/{col}.png", width=8, height=6, dpi=80, verbose=False)
sorted(os.listdir("eda_plots"))

In [ ]:
mlflow.set_experiment("income-classifier")

with mlflow.start_run(run_name="eda") as run:
    raw_uri = upload_df(df, "mlflow-datasets", "income-classifier/raw/income_data.csv")
    mlflow.log_input(mlflow.data.from_pandas(df, source=raw_uri, name="income_data"), context="raw")
    mlflow.log_param("rows", len(df))
    mlflow.log_artifacts("eda_plots", artifact_path="eda_plots")

print("run_id :", run.info.run_id)
print("raw    :", raw_uri)
print("artifacts:", [a.path for a in mlflow.MlflowClient().list_artifacts(run.info.run_id, "eda_plots")])